In [0]:
#workforce_fte_org code 
 
from pyspark.sql.functions import (col,trim,when,to_date,lit,current_timestamp,lower, substring)  
 
bronze_workforce_fte_org_df= (   
	spark.read   
	.option("header", True)   
	.option("inferSchema", True)   
	.csv("abfss://data@jdnhsbronze.dfs.core.windows.net/dbo.jdnhs_workforce_fte_org.csv")   
	)   

bronze_workforce_fte_org_df.show(10)   
bronze_workforce_fte_org_df.printSchema()  
 
  
# check schema here and if its wrong add code to make it correct data types   
#month already done   
 


In [0]:
# rename columns 
bronze_workforce_fte_org_df = (   
	bronze_workforce_fte_org_df   
	.withColumnRenamed("data_month", "month") 
	.withColumnRenamed("nhse_region_name", "region_name") 
	.withColumnRenamed("nhse_region_code", "region_code") 
	.withColumnRenamed("staff_group_category", "staff_group")  
	)   	

	
print(bronze_workforce_fte_org_df.columns)

In [0]:
# standardise data 
  
silver_workforce_fte_org_df = (bronze_workforce_fte_org_df   
	.withColumn(
        "month",to_date(
            substring(trim(col("month")), 1, 10),"yyyy-MM-dd"))
	.withColumn("data_type", lower(trim(col("data_type"))))  
	.withColumn("region_name", lower(trim(col("region_name")))) 
	.withColumn("region_code", lower(trim(col("region_code")))) 
	.withColumn("ics_name", lower(trim(col("ics_name")))) 
	.withColumn("ics_code", lower(trim(col("ics_code")))) 
	.withColumn("org_name", lower(trim(col("org_name")))) 
	.withColumn("org_code", lower(trim(col("org_code")))) 
	.withColumn("staff_group", lower(trim(col("staff_group")))) 
	.withColumn("staff_group_label", lower(trim(col("staff_group_label")))) 
	.withColumn("fte", (trim(col("fte")))) 
	.withColumn("source_file", lower(trim(col("source_file"))))   
	.withColumn("source_sheet", lower(trim(col("source_sheet"))))     
	)   

# clean data 

silver_workforce_fte_org_df = (silver_workforce_fte_org_df   
		.withColumn(   
		"fte",   
	    when(col("fte") == ".",None).otherwise(   
	        col("fte")).cast("double")  
	    )
       
  .withColumn("month",to_date(col("month"),"yyyy-MM-dd"))   
)  
 
# check data 
silver_workforce_fte_org_df.show(10)   
  
  


In [0]:
silver_workforce_fte_org_df.select("region_name").distinct().show(10) 

In [0]:
#validate data  
valid_workforce_fte_org_df = silver_workforce_fte_org_df.filter(   
    (col("data_type") == "fte") &  
    col("staff_group").isNotNull() &  
    col("staff_group_label").isNotNull() &  
	  (   
        col("fte").isNull() |   
        (col("fte") >= 0)  
    )  & 
	  col("source_file").isNotNull() &  
		col("source_sheet").isNotNull() &  
    col("load_timestamp").isNotNull()  
   )
quarantine_workforce_fte_org_df = silver_workforce_fte_org_df.filter(   
  (col("data_type")!= "fte") |  
   col("staff_group").isNull() |   
    col("staff_group_label").isNull() | 
	  (   
        col("fte").isNotNull() &   
        (col("fte") < 0)  
    ) |
	  col("source_file").isNull() |   
		col("source_sheet").isNull() |   
    col("load_timestamp").isNull()  
 )

 # check data
print("Number of Quarantine Rows: ",
quarantine_workforce_fte_org_df.count())
print("Number of Valid Rows: ", valid_workforce_fte_org_df.count())



In [0]:

#show actual valid and quarantine tables

quarantine_workforce_fte_org_df.show()
valid_workforce_fte_org_df.show()



In [0]:


# write data 



(valid_workforce_fte_org_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "path",
        "abfss://silver-tables@jdnhsbronze.dfs.core.windows.net/workforce_fte/"
    ) \
    .saveAsTable(
        "silver_workforce_fte"
    ))

# Invalid data → Quarantine
(
    quarantine_workforce_fte_org_df.write
    .format("delta")
    .mode("overwrite")
    .save(
        "abfss://quarantine@jdnhsbronze.dfs.core.windows.net/workforce_fte/"
    )
)
